# **Part 5: Differential Expression Analysis**

---

## **Table of Contents**

- [Preliminary Setup](#preliminary-setup)
- [Import Packages](#import-packages)
- [Read and Prepare VST Data](#read-and-prepare-vst-data)
- [Expression Heatmaps Coupled with Hierarchical Clustering](#expression-heatmaps-coupled-with-hierarchical-clustering)
- [Transitioning from EDA to Differential Expression Analysis](#transitioning-from-eda-to-differential-expression-analysis)
- [Building Mathematical Intuition](#building-mathematical-intuition)
- [Formal Differential Expression Analysis (DEA)](#formal-differential-expression-analysis-dea)
- [The Multiple Testing Problem in Transcriptomics](#the-multiple-testing-problem-in-transcriptomics)
  - [Bonferroni Correction](#bonferroni-correction)
  - [Controlling the False Discovery Rate](#controlling-the-false-discovery-rate)
- [Statistical Testing with PyDESeq2](#statistical-testing-with-pydeseq2)
- [Volcano Plot for DEGs](#volcano-plot-for-degs)
- [Interpretation](#interpretation)
- [Next Steps](#next-steps)
- [Summary](#summary)

---

## **Preliminary Setup**

In [ ]:
# automatically re-import custom modules
%reload_ext autoreload
%autoreload 2

%matplotlib inline

In [ ]:
from pathlib import Path
import sys

# Resolve the absolute root directory of the project
project_root = Path.cwd().parent.resolve()

# Prepend project root to Python's search path to prioritize local module imports
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Define the directory path for intermediate outputs
intermediate_dir = project_root / "results" / "intermediates"
intermediate_dir.mkdir(parents=True, exist_ok=True)

# Define figures directory
figs_dir = project_root / "results" / "figures"
figs_dir.mkdir(parents=True, exist_ok=True)

# Define absolute system paths for the CCLE data subsets
counts_filepath = intermediate_dir / "1_ccle_counts_subset.csv"
meta_filepath = intermediate_dir / "1_ccle_meta_subset.csv"

---

## **Import Packages**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage
from scipy.cluster.hierarchy import fcluster
from PyComplexHeatmap import ClusterMapPlotter, HeatmapAnnotation, anno_simple, anno_label
from sklearn.decomposition import PCA
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats
from bioinfokit import visuz # not actively maintained

# Local imports
from src.utils.helpers import prepare_vst_data, plot_volcano

---

## **Read and Prepare VST Data**

In the previous [notebook](./3_pca_analysis.ipynb), we saved the variance-stabilized transformation (VST) data as an `.h5ad` file (`1_ccle_vst_processed.h5ad`). For first part of this notebook, we will simply read this pre-computed object back into memory. Alternatively, we could regenerate the `dds` object from scratch by rerunning the pipeline steps sourced by `prepare_vst_data` helper function and the original `ccle_counts_subset` and `ccle_meta_subset` dataframes as its inputs.

In the [second part](#transitioning-from-eda-to-differential-expression-analysis) of the notebook, we will run a sligtly different pipeline without applying the VST. Additionally, we will be using a specific condition for the "design" formula.

In [ ]:
# Define caching path and execution flag
vst_filepath = intermediate_dir / '1_ccle_vst.h5ad'
use_cached_data = True 

# Load cached VST data if available and requested; otherwise, compute from scratch
if vst_filepath.exists() and use_cached_data:
    print(f"Loading cached DeseqDataSet from: {vst_filepath}")
    dds = sc.read_h5ad(vst_filepath)
else:
    print("Cache missing or stale. Running PyDeseq2 VST pipeline...")
    dds = prepare_vst_data(counts_filepath, meta_filepath, vst_filepath=vst_filepath)

Let's verify the `dds` object and create `vst_df`:

In [ ]:
print(dds.shape)
print(dds)

In [ ]:
# Extract the transformed VST counts into a structured pandas DataFrame for analysis
vst_df = pd.DataFrame(
    dds.layers['vst_counts'],
    index=dds.obs_names,   # Sample IDs
    columns=dds.var_names  # Gene Names
)
vst_df.head()

---

## **Expression Heatmaps Coupled with Hierarchical Clustering**

Like we did in the [previous notebook](./4_hierarchical_clustering.ipynb), let's regenerate the heatmap showing hierarchical clustering results using the 500 most highly variable genes and three clusters ($k = 3$) and annotate the samples in the heatmap with `tcga_code` histologies:

In [ ]:
# Subset top 500 most variable genes
top_genes = vst_df.var(axis=0).nlargest(500).index
vst_subset = vst_df[top_genes]

# Compute explicit pairwise Euclidean distance matrix for samples
sample_distances = pdist(vst_subset, metric='euclidean')

# Compute linkages for samples
sample_linkage = linkage(sample_distances, metric='euclidean', method='complete')

# Compute clusters for samples
sample_cluster_assignments = fcluster(sample_linkage, t=3, criterion='maxclust')
sample_clusters = pd.Series(sample_cluster_assignments.astype(str), index=vst_subset.index)

# Define column annotations
col_ann = HeatmapAnnotation(
    Cluster=anno_label(
        # cluster_labels,
        sample_clusters,
        merge=True,             # Merges adjacent labels so you only see one label per block
        rotation=0,             
    ),
    TCGA_code=anno_simple(
        dds.obs['tcga_code'],
        cmap='Set1',
        legend=True
    ),
    label_side='right'
)

# Plot heatmap
fig = plt.figure(figsize=(8, 12))
cm = ClusterMapPlotter(
    data=vst_subset.T,
    top_annotation=col_ann,
    bottom_annotation=None,
    left_annotation=None,
    right_annotation=None,
    row_cluster=True,
    col_cluster=True,
    row_cluster_method='complete',
    row_cluster_metric='euclidean',
    col_cluster_method='complete',
    col_cluster_metric='euclidean',
    show_rownames=False,
    show_colnames=False,
    row_names_side='right',
    col_names_side='bottom',
    row_dendrogram=True,
    col_dendrogram=True,
    row_dendrogram_size=10,
    col_dendrogram_size=10,
    row_split=None,
    col_split=sample_clusters,
    tree_kws=None,
    row_split_order=None,
    col_split_order=None,
    row_split_gap=0.5,
    col_split_gap=1,
    mask=None,
    legend=True,
    legend_order='auto',
    legend_side='right',
    cmap='bwr',
    label='VST Expression',
    xlabel='Samples',
    ylabel='Genes',
    xlabel_side='bottom',
    ylabel_side='right',
    verbose=0,
)

fig.savefig(figs_dir / "hierarchical_clustering_heatmap.png", bbox_inches='tight')

Let's visualize these cluster assignments on the PCA $PC1-PC2$ plot:

In [ ]:
# Compute first two Principal components 
pca = PCA(n_components=2)
pca_compoments = pca.fit_transform(vst_subset)

# Create a frame for first two PCs
pca_df = pd.DataFrame(
    pca_compoments,
    index=vst_subset.index,
    columns=["PC1", "PC2"]
)

# Compute explained variance ratios (percentage)
explained_var_ratios = pca.explained_variance_ratio_ * 100


# Assign clusters computed from h-clustering
pca_cluster_df = pca_df.assign(Cluster=sample_clusters)


# Plot samples on PC1-PC2 axes
fig, ax = plt.subplots(figsize=(5, 4))

sns.scatterplot(ax=ax, data=pca_cluster_df, x="PC1", y="PC2", hue="Cluster", s=50, alpha=0.5, palette='Set1')
ax.set_xlabel(f"PC1: {explained_var_ratios[0]:0.2f}% variance")
ax.set_ylabel(f"PC2: {explained_var_ratios[1]:0.2f}% variance");

# Save figure
fig.savefig(figs_dir / "PCA_clusters.png", bbox_inches='tight')

Our clustering results demonstrate a clear split: Cluster 1 is predominantly composed of **SCLC** samples, while Clusters 2 and 3 consist almost entirely of **LUAD** and **LUSC** samples.

This strict partitioning is rooted in the distinct developmental biology of these malignancies, which we previously explored during our PCA workflow. Lung cancer is broadly categorized into non-small cell lung cancer (NSCLC) and small cell lung cancer (SCLC):

* **LUAD and LUSC (NSCLC):** Both subtypes arise from lung epithelial cells, specifically, alveolar cells and bronchial basal cells, respectively. Despite their histopathological differences, they share a common epithelial lineage and baseline transcriptional program.
* **SCLC:** Unlike the epithelial NSCLC subtypes, SCLC originates from pulmonary neuroendocrine cells.

This fundamental difference in cellular origin is the primary driver of the global transcriptomic split. Because SCLC belongs to a neuroendocrine lineage rather than an epithelial one, its gene expression profile is profoundly distinct. This developmental divergence explains why LUAD and LUSC samples cluster together, while SCLC segregates into a completely independent branch.

For a deeper dive into these cellular lineages, please refer to the [Lung Cancer Biology Primer](../bonus/docs/lung_cancer_biology_primer.md).

---

## **Transitioning from EDA to Differential Expression Analysis**

A primary objective of transcriptomic profiling is to systematically identify gene expression differences between distinct experimental or clinical groups. Throughout our EDA work in the previous notebooks, we established that the unsupervised topological patterns, uncovered by both PCA and hierarchical clustering, strongly align with three main historical TCGA codes, predominantly composed of neuroendocrine **SCLC** samples, while Clusters 2 and 3 are composed of epithelial **LUAD** and **LUSC** samples.

While our visual workflows allowed us to hand-pick a few obvious candidate genes driving these splits, a manual search cannot scale across the entire transcriptome. To systematically map the transcriptional variance across all genes, we need a robust statistical framework. Differential Expression Analysis (DEA) using `DESeq2` or `PyDESeq2` is designed to do exactly this: scan the entire transcriptome to uncover all genes that show statistically significant differences in expression between our defined sample groups. In the following sections, we will deep dive into it.

### **Building Mathematical Intuition**

Before diving into the formal `PyDESeq2` statistical pipeline, we will perform a simplified, "back-of-the-envelope" version of this analysis using our original variance-stabilized (`vst_df`) expression dataframe.

Deconstructing the core concepts behind these calculations provides valuable intuition for how the underlying mathematical models evaluate variance log-fold changes behind the scenes to generate a final list of Differentially Expressed Genes (DEGs).

To keep our calculation simpler, we'll combine cluster 2 and 3 to end up having two major clusters and map them to their corresponding TCGA codes:

In [ ]:
# Map cluster 1 to `SCLC` and clusters 2 and 3 to `LUAD/LUSC`
major_sample_clusters = sample_clusters.map({'1': 'SCLC', '2': 'LUAD/LUSC', '3' : 'LUAD/LUSC'})

# Verify presence of two clusters
print(major_sample_clusters.value_counts())

In the next step, we will find the mean of variance-stabilized counts for each gene per cluster and calcuate the fold change for every single gene between **SCLC** (cluster 1) and **LUAD/LUSC** (cluster 2). Subsequently, we will take the $log2$ of that fold change ($log2FC$). Note that $log2FC$ is computed with a pseudo-count of 1 to avoid division by zero:

In [ ]:
# Add cluster information to `vsd_df` dataframe
vst_with_clusters = vst_df.assign(Cluster=major_sample_clusters)
vst_with_clusters.head()

In [ ]:
# Comopute mean of each gene across samples per cluster
cluster_means = vst_with_clusters.groupby("Cluster").apply(lambda df: df.mean(axis=0))

# Preview cluster means
cluster_means.head()

In [ ]:
# Take log2 of clutser 1 mean and cluster 2 mean ratio
log2FC = np.log2((cluster_means.loc['SCLC'] + 1) / (cluster_means.loc['LUAD/LUSC'] + 1))
print(log2FC)

Let's create a histogram plot of log2FC values:

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))

sns.histplot(ax=ax, data=log2FC, bins=30, color='skyblue')
ax.set_title("Log2FC Distribution", fontweight='bold')
ax.set_xlabel("Log2FC")
ax.set_ylabel("Frequency");

The histogram shows the distribution of log2FC values. Most genes are not differentially expressed and so have a log2FC value near 0. Very few genes have a log2FC value of less than -0.5 or greater than 0.5. These are likely our most differentially expressed genes.

To identify DEGs, we will create a dataframe, `degs_df` with gene names, mean expression of those genes, and log2FC values:

In [ ]:
degs_df = pd.DataFrame(
    {
        'Mean Expression': cluster_means.mean(axis=0),
        'Log2FC': log2FC
    }
)

print(degs_df.shape)

We then print top 50 highly DEGs, both up-regulted and down-refulated ones:

In [ ]:
# Sort `degs_df` by its `Log2FC` column absolute values and extract gene names
degs_ranks = degs_df['Log2FC'].abs().sort_values(ascending=False).index

# Extract top 50 DEGs
top_degs = degs_df.loc[degs_ranks].head(50)

# Preview
top_degs

In [ ]:
# Print top 10 up-regulated genes in cluster 1
top_degs_up = degs_df.sort_values("Log2FC", ascending=False).head(10)
print("Top 10 up-regulated DEGs:\n", top_degs_up)

# Print top 10 down-regulated genes in cluster 1
top_degs_down = degs_df.sort_values("Log2FC", ascending=True).head(10)
print("\nTop 10 down-regulated DEGs:\n", top_degs_down)

Now, let's plot the expression of top 3 DEGs that are higher in cluster 1 than cluster 2 ($log2FC \gt 0$) and vice verca in the PCA plot and see if these genes are differentially expressed.

In [ ]:
# Extract VST counts for top six degs 
degs = top_degs_up.head(3).index.tolist() + top_degs_down.head(3).index.tolist()
degs_vst = vst_df[degs]

# Merge PCA coordinates with VST expression counts fot top DEGs
pca_df_degs = pca_df.merge(degs_vst, on='CCLE_ID')

# Convert `pca_df_degs` to a long dataframe
pca_df_degs_long = pd.melt(
    pca_df_degs,
    id_vars=['PC1', 'PC2'],
    value_vars=degs,
    var_name='Gene',
    value_name='Expression (VST)'
)

# Plotting PC1 vs PC2 faceted by Gene.
g = sns.relplot(
    data=pca_df_degs_long,
    x='PC1', y='PC2',
    col='Gene', col_wrap=3,
    hue='Expression (VST)',
    palette=sns.blend_palette(["blue", "red"], as_cmap=True),
    alpha=0.75,
    kind="scatter",
    s=50,
    height=3,     # Height of EACH subplot (in inches)
    aspect=1.1,   # Width of each subplot will be height * aspect 
)

# Formatting
g.set_axis_labels(
    f"PC1: {explained_var_ratios[0]:.1f}% variance",
    f"PC2: {explained_var_ratios[1]:.1f}% variance",
    fontsize=14)
g.set_titles(col_template="{col_name}", size=18, weight='bold')
for ax in g.axes.flat:
    ax.tick_params(axis='both', labelsize=12,)
plt.setp(g._legend.get_texts(), fontsize='14') 
plt.setp(g._legend.get_title(), fontsize='14');

# Save figure
g.savefig(figs_dir / "top_vst_deg_pca.png", dpi=300, bbox_inches='tight')

Remarkably, even a simplified, back-of-the-envelope calculation on our variance-stabilized matrix is sufficient to identify differentially expressed genes separating Cluster 1 from Cluster 2.

>**Note**: As established in our earlier EDA, variance-stabilized counts are optimized for low-dimensional visualization, such as PCA and hierarchical clustering, rather than formal differential expression analysis. We temporarily relaxed this rule here solely as a conceptual exercise to build intuition, prior to applying PyDESeq2's native size-factor normalization and negative binomial modeling.

### **Formal Differential Expression Analysis (DEA)**

In our initial exploratory setups, we assigned a placeholder design formula (`design = ~ Pathology`) simply to satisfy the constructor's requirements. Now, having established through thorough EDA that our samples separate cleanly into distinct molecular lineages, we can define a far more meaningful experimental design: `design = ~ Cluster`. This allows us to directly extract the key driver genes separating neuroendocrine **SCLC** cell lines from epithelial **LUAD/LUSC** lineages.

While our preliminary visualizations relied on variance-stabilized values, the core statistical engine of `PyDESeq2` operates directly on raw counts, adjusting for library depth using median-of-ratios size-factor normalization. The primary modeling and normalization steps are executed seamlessly through a single wrapper method, `deseq2()`. This single call handles size-factor estimation, dispersion parameter fitting, and negative binomial GLM testing across the entire transcriptome.

To prepare our dataset for the PyDESeq2 workflow, we will aggregate our samples into two primary cohorts, the **SCLC** cluster and the combined **LUAD/LUSC** cluster, and append these group designations to our sample metadata. As a reminder, we will still use VST data for clustering. Then, We will pass the raw count matrix alongside this updated metadata object to initialize and execute `deseq2()`:

In [ ]:
# `vst_subset` holds VST data for top 500 most variable genes.
# We will create three clusters and combine cluster 1 and 2
sample_linkage = linkage(pdist(vst_subset, metric='euclidean'), method='complete', metric='euclidean')
sample_clusters_assignments = fcluster(sample_linkage, t=3, criterion='maxclust')
sample_clusters = pd.Series(sample_cluster_assignments.astype(str), index=vst_subset.index)
sample_clusters = sample_clusters.map({'1': 'SCLC', '2': 'LUAD/LUSC', '3': 'LUAD/LUSC'})

# Load original count matrix and transpose it for `DeseqDataSet`
counts_subset = pd.read_csv(counts_filepath, header=0, index_col=0)
counts_subset_T = counts_subset.transpose(copy=True).astype(int)

#  Load the associated sample metadata and add `Cluster` column
meta_subset = pd.read_csv(meta_filepath, header=0, index_col=0)
meta_subset_with_clusters = meta_subset.assign(Cluster=sample_clusters)

# Initialize the DeseqDataSet with a Cluster-based experimental design
inference = DefaultInference(n_cpus=None)
dds = DeseqDataSet(
    counts=counts_subset_T,
    metadata=meta_subset_with_clusters,
    design="~Cluster",
    refit_cooks=True,  
    inference=inference,
)

# Run deseq2
dds.deseq2(fit_type='parametric')

Let's do some exploratory analysis: compare "raw counts" vs "normalized counts" corrected for sequencing depth (or library size).

In [ ]:
# Create a dataframe fot normalized counts
norm_counts_df = pd.DataFrame(
    dds.layers['normed_counts'],
    index=counts_subset_T.index,
    columns=counts_subset_T.columns
)

# Choose a random gene as GOI
goi = np.random.choice(counts_subset.index, size=1).item()

# Create a dataframe for GOI with raw counts, norm. counts 
# and size factors across all samples
goi_df = pd.DataFrame({
    'Raw counts': counts_subset_T[goi],
    'Normed counts': norm_counts_df[goi],
    'Size factor': dds.obs['size_factors']
})

# Plot Raw counts against norm. counts colored based on size factors
fig, ax = plt.subplots(figsize=(7,5))
sns.scatterplot(ax=ax, data=goi_df, x='Raw counts', y='Normed counts', hue='Size factor', palette='plasma', alpha=0.75)
ax.set_title(goi, fontweight='bold')

max_val = goi_df[['Raw counts', 'Normed counts']].max().max().item()
ax.plot([0, max_val], [0, max_val]);

This relationship aligns directly with our expectations: samples with a $\text{size factor} > 1$ (positioned below the identity line) are adjusted downward, whereas samples with a $\text{size factor} < 1$ (positioned above the identity line) are scaled upward.

### **The Multiple Testing Problem in Transcriptomics**

In high-throughput RNA-seq DEA, statistical testing is performed independently across every measured gene. If a dataset contains 10,000 genes, we execute 10,000 distinct hypothesis tests, generating 10,000 corresponding $p$-values.

This scale introduces a significant statistical challenge: the multiple testing problem. Even in a null scenario where no true biological differences exist between Cluster 1 and Cluster 2, evaluating thousands of hypotheses simultaneously dramatically inflates the probability of false discoveries (Type I errors).

Using a standard significance threshold of $\alpha = 0.05$, we would expect approximately 500 false positives purely by chance:

$$10,000 \times 0.05 = 500$$

Relying on unadjusted $p$-values means that a large fraction of our "significant" candidate genes would simply be stochastic noise rather than true differentially expressed transcripts.

#### **Bonferroni Correction**

One classic approach to address multiple testing is the Bonferroni correction, which adjusts individual $p$-values by multiplying them directly by the total number of tests ($N$):

$$p_{\text{adjusted}} = \min(1, \, p \times N)$$

For example, if a gene yields a raw $p$-value of $0.00001$, its Bonferroni-adjusted value across 10,000 tests becomes:

$$p_{\text{adjusted}} = 0.00001 \times 10,000 = 0.1$$

Under this adjustment, a result that initially appeared strong no longer meets the standard significance threshold. While the Bonferroni method strictly controls the Family-Wise Error Rate (FWER), the probability of making *at least one* false positive, it is notoriously over-conservative for transcriptomic data. By setting an excessively high bar for significance, it severely reduces statistical power and eliminates many true biological signals (false negatives).

#### **Controlling the False Discovery Rate**

Rather than controlling the probability of making a single false positive across the entire genome, a more practical framework controls the False Discovery Rate (FDR): the expected proportion of false positives among all genes declared significant.

Setting an FDR threshold of $\text{FDR} < 0.05$ ensures that, on average, no more than 5% of our reported differentially expressed genes are false discoveries. This approach creates a balance between controlling false positives and preserving statistical power to detect real biological changes.

To control the FDR, `PyDESeq2` applies the Benjamini-Hochberg (BH) procedure by default. This step ranks all raw $p$-values in ascending order and adjusts them sequentially based on their rank:

$$p_{\text{adj}}^{(i)} = \min \left( p_{(i)} \times \frac{m}{i}, \, 1 \right)$$

where $i$ is the rank of the $p$-value, $m$ is the total number of tests, and $p_{(i)}$ is the $i$-th smallest $p$-value. By scaling $p$-values according to their relative rank, the Benjamini-Hochberg adjustment maintains strict control over false discoveries while avoiding the over-penalization typical of Bonferroni correction.

If you're interested in the topic, check out my [blog](https://paymantohidifar.github.io/blogs/statistical_tests.html), where I have reviewed several common statistical tests.

#### **Statistical Testing with PyDESeq2**
We will use `DeseqStats.summary()` to execute Wald tests across all genes, computing $\log_2 \text{FC}$ values alongside their corresponding raw $p$-values and BH-adjusted $p$-values (`padj`).

Note that in this analysis, we will set **Cluster 1: SCLC** as our tested cohort and **Cluster 2: LUAD/LUSC** as our reference baseline. This is simply done through the `contrast` argument when we initialize `DeseqStats` object:

In [ ]:
# Initialize DeseqStats with desired FDR cutoff (alpha)
stat_res = DeseqStats(
    dds=dds,
    contrast=('Cluster', 'SCLC', 'LUAD/LUSC'),  # (factor, tested, reference)
    alpha=0.05,                  # Target FDR threshold (5%)
    independent_filter=True,      # Enables automatic low-count filtering
    inference=inference,
    quiet=True
)

# Run Wald tests and calculate adjusted p-values (padj)
stat_res.summary()

# Extract the results DataFrame
results_df = stat_res.results_df

# Preview results dataframe
results_df.head()

Let's try to preview the most striking DEGs. To do that, we will select only the genes with $padj < 0.001$, then reorder the `results_df` from greatest to least absolute `log2FoldChange`:

In [ ]:
fdr_threshold = 0.001

# Filter for genes meeting the FDR threshold
top_degs = results_df[
    results_df['padj'] < fdr_threshold
]['log2FoldChange'].abs().sort_values(ascending=False).index

top_degs_df = results_df.loc[top_degs]

print(f"Found {len(top_degs)} DEGs at FDR < {fdr_threshold}")

top_degs_df.head(10)

Let's plot the expression of top DEGs in a PCA plot as above to gut-check whether these DEGs make sense:

In [ ]:
# Extract VST counts for top six degs 
degs = top_degs[:6]
degs_vst = vst_df[degs]

# Merge PCA coordinates with VST expression counts fot top DEGs
pca_df_degs = pca_df.merge(degs_vst, on='CCLE_ID')

# Convert `pca_df_degs` to a long dataframe
pca_df_degs_long = pd.melt(
    pca_df_degs,
    id_vars=['PC1', 'PC2'],
    value_vars=degs,
    var_name='Gene',
    value_name='Expression (VST)'
)

# Plotting PC1 vs PC2 faceted by Gene.
g = sns.relplot(
    data=pca_df_degs_long,
    x='PC1', y='PC2',
    col='Gene', col_wrap=3,
    hue='Expression (VST)',
    palette=sns.blend_palette(["blue", "red"], as_cmap=True),
    alpha=0.75,
    kind="scatter",
    s=50,
    height=3,     # Height of EACH subplot (in inches)
    aspect=1.1,   # Width of each subplot will be height * aspect 
)

# Formatting
g.set_axis_labels(
    f"PC1: {explained_var_ratios[0]:.1f}% variance",
    f"PC2: {explained_var_ratios[1]:.1f}% variance",
    fontsize=14)
g.set_titles(col_template="{col_name}", size=18, weight='bold')
for ax in g.axes.flat:
    ax.tick_params(axis='both', labelsize=12,)
plt.setp(g._legend.get_texts(), fontsize='14') 
plt.setp(g._legend.get_title(), fontsize='14');

### **Volcano Plot for DEGs**

With our PyDESeq2 differential expression analysis complete, we can now synthesize our transcriptome-wide findings into a single volcano plot. Volcano plots remain the standard figure for presenting gene expression results, and no differential expression pipeline is quite complete without one. Let's walk through how to construct one to highlight our top candidate genes.

We will construct our volcano plot using a custom helper function, `plot_volcano`. Alternatively, the `bioinfokit` [package](https://pypi.org/project/bioinfokit/2.1.4/) offers a convenient out-of-the-box option via `visuz.GeneExpression.volcano`. While both methods yield publication-ready figures, using a custom helper function provides full flexibility over aesthetic parameters, label repelling, and threshold highlighting.

In [ ]:
fig, ax = plot_volcano(
    df=results_df,
    lfc_col='log2FoldChange', 
    padj_col='padj', 
    gene_col='Gene',
    lfc_thresh=2.0, 
    padj_thresh=0.01,
    top_n_labels=10,
    title="Volcano Plot:\nSCLC (Cluster 1) vs LUAD/LUSC (Cluster 2)",
    figsize=(10, 6)
)

Let's refine the volcano plot by labeling only the top 10 upregulated and top 10 downregulated DEGs (ranked by absolute $\log_2 \text{FoldChange}$):

In [ ]:
top_degs_up = results_df['log2FoldChange'].sort_values(ascending=False).head(10).index.tolist()
top_degs_down = results_df['log2FoldChange'].sort_values(ascending=True).head(10).index.tolist()

genes_to_label = top_degs_up + top_degs_down

fig, ax = plot_volcano(
    df=results_df,
    lfc_col='log2FoldChange', 
    padj_col='padj', 
    gene_col='Gene',
    lfc_thresh=2.0, 
    padj_thresh=0.01,
    select_labs=genes_to_label,
    title="Volcano Plot:\nSCLC (Cluster 1) vs LUAD/LUSC (Cluster 2)",
    figsize=(10, 6)
)

# Save figure
fig.savefig(figs_dir / "Volcano_plot.png", dpi=300, bbox_inches='tight')

As noted earlier, we can also generate this plot using `visuz.GeneExpression.volcano` from the `bioinfokit` package. However, because this function does not automatically handle `NaN` values in the $\log_2 \text{FoldChange}$ or adjusted $p$-value columns, we must filter out unquantified genes prior to plotting:

In [ ]:
# Drop genes (rows) with NaN values on `log2FoldChange`
results_df_clean = results_df.dropna(subset=["log2FoldChange", "padj"], axis=0)
results_df_clean.reset_index(inplace=True)

print(
    f"Lost {np.round((len(results_df) - len(results_df_clean)) / len(results_df) * 100, 2)} % of genes due to NaN values."
    )

# Generates a volcano plot directly from a Pandas DataFrame
visuz.GeneExpression.volcano(
    df=results_df_clean, 
    lfc='log2FoldChange', 
    pv='padj', 
    geneid='Gene',
    genenames=tuple(genes_to_label),
    axtickfontname="Serif",
    axlabelfontname="Serif",
    lfc_thr=(2.0, 2.0), 
    pv_thr=(0.01, 0.01),
    sign_line=True,
    color=("crimson", "grey", "navy"),
    plotlegend=True,
    legendpos='upper right',
    dim=(10, 6),
    theme='light',
    figname="Volcano Plot:\nSCLC (Cluster 1) vs LUAD/LUSC (Cluster 2)",
    show=True
)

### **Interpretation**

SCLC samples upregulate neurosecretory machinery (GRP, PCSK2), synaptic transport (SLC17A6), and neuronal structural factors (DCX, NRSN1).

NSCLC samples (LUAD/LUSC) downregulate those neuroendocrine features and upregulate epithelial integrity markers (IVL, SFTA2), inflammation mediators (LCN2, CXCL5), and oncofetal antigens (PSG family).

---

## **Next Steps**

While a volcano plot gives a great high-level view of individual top hits (comparing effect size against statistical significance), it doesn't tell us what those genes actually do as a group.

To go beyond handful-genes interpretation, transcriptomic analysis relies on several complementary downstream workflows to interpret hundreds or thousands of DEGs simultaneously:

**Functional Enrichment & Pathway Analysis**: Enrichment methods group DEGs into known biological categories to test if specific pathways are statistically overrepresented.

* **Over-Representation Analysis (ORA)**: Tests whether our list of significant DEGs contains more genes belonging to a specific category than expected by chance (using hypergeometric or Fisher's exact tests).

* **Gene Ontology (GO)**: Categorizes genes into Biological Process, Molecular Function, and Cellular Component.
* **KEGG / Reactome:** Maps DEGs onto curated metabolic and signaling pathways.
* **Gene Set Enrichment Analysis (GSEA)**: Unlike ORA (which uses a hard threshold like $\text{padj} < 0.05$), GSEA ranks all genes by their $t$-statistic or $\log_2 \text{FC}$ and evaluates whether predefined gene sets are enriched at the top or bottom of the ranked list. This is ideal for detecting subtle, coordinated pathway shifts.

In next notebook, we will explore some of these downstream analyses.

---

## **Summary**

We began our analysis by running PCA to uncover the dominant drivers of transcriptomic variation across the CCLE lung cancer dataset. Unsupervised hierarchical clustering validated these patterns, confirming that the cell lines naturally partition into distinct, highly concordant groups.

To understand what distinguishes these cohorts, we ran differential expression analysis with PyDESeq2. Interpreting these candidate markers revealed a clear biological distinction: neuroendocrine expression profiles in SCLC versus epithelial barrier signatures in LUAD/LUSC. Having mapped these core biological signals, we now have a robust foundation to investigate specific follow-up questions and test deeper mechanistic hypotheses.